In [ ]:
# Cell 1: Data Loading and Filter Setup
# This cell imports the required libraries and loads the AAC shelter
# outcomes dataset from a CSV file into a Pandas DataFrame.
# It also defines the filter_map dictionary which maps rescue category
# labels to pre-filtered DataFrames for use in the dashboard callbacks.
#
# Note: The original implementation loads data from CSV rather than
# from MongoDB through the CRUD module. All three rescue filters
# use the same condition (animal_type == 'Dog') as placeholders.
#
# Author: Ethan Chapman
# Course: CS 340 - Client/Server Development

import pandas as pd              # Data manipulation and analysis
import plotly.express as px       # Interactive chart creation
from dash import Dash, dcc, html, Input, Output, dash_table  # Dash web framework components

# Load the Austin Animal Center shelter outcomes dataset from CSV
df_csv = pd.read_csv("aac_shelter_outcomes (1).csv")

# Filter mapping dictionary
# Maps each rescue type label to a filtered DataFrame
# Currently all three rescue types use the same filter (dogs only)
filter_map = {
    "Reset": df_csv,                                                    # Show all records
    "Water Rescue": df_csv[df_csv["animal_type"] == "Dog"],             # Placeholder - needs specific breed/sex/age criteria
    "Mountain or Wilderness Rescue": df_csv[df_csv["animal_type"] == "Dog"],  # Placeholder - needs specific breed/sex/age criteria
    "Disaster Rescue or Individual Tracking": df_csv[df_csv["animal_type"] == "Dog"],  # Placeholder - needs specific breed/sex/age criteria
}


In [ ]:
# Cell 2: Initialize the Dash Application
# Create the Dash web application instance that will serve the dashboard
app = Dash(__name__)


In [ ]:
# Cell 3: Dashboard Layout
# This cell defines the visual structure of the dashboard using Dash
# HTML and core components. The layout includes:
#   - A title and SNHU branding bar with author attribution
#   - A dropdown menu for selecting rescue type filters
#   - A range slider for filtering animals by age in weeks
#   - An interactive data table showing filtered records
#   - A pie chart for animal type distribution
#   - A geolocation scatter map showing animal locations

app.layout = html.Div([
    # Dashboard title
    html.H1("AAC Shelter Outcomes Dashboard"),
    
    # Branding section with SNHU logo and author name
    html.Div([
        html.A(html.Img(src="https://upload.wikimedia.org/wikipedia/commons/1/1f/SNHU_Logo.png", height="50px"),
               href="https://www.snhu.edu"),
        html.Span("Dashboard by Ethan Chapman", style={"margin-left": "20px", "font-weight": "bold"})
    ], style={"display": "flex", "align-items": "center"}),
    
    # Filter controls section
    html.Div([
        # Rescue type dropdown filter
        html.Label("Select Rescue Filter:"),
        dcc.Dropdown(
            id='filter-dropdown',
            options=[{"label": k, "value": k} for k in filter_map.keys()],  # Build options from filter_map keys
            value="Reset",        # Default selection shows all records
            clearable=False       # Prevent clearing the selection
        ),
        html.Br(),
        # Age range slider for filtering by age in weeks
        html.Label("Filter by Age (weeks):"),
        dcc.RangeSlider(
            id='age-slider',
            min=0,                                                # Minimum age value
            max=df_csv['age_upon_outcome_in_weeks'].max(),         # Maximum age from dataset
            step=1,                                                # Increment by 1 week
            value=[0, df_csv['age_upon_outcome_in_weeks'].max()],  # Default: full range
            marks={0: '0', int(df_csv['age_upon_outcome_in_weeks'].max()): str(int(df_csv['age_upon_outcome_in_weeks'].max()))},
            tooltip={"placement": "bottom", "always_visible": True}  # Show current values
        )
    ], style={"width": "50%", "margin-bottom": "20px"}),
    
    # Interactive data table displaying filtered animal records
    dash_table.DataTable(
        id='data-table',
        columns=[{"name": i, "id": i} for i in df_csv.columns],  # Create column definitions from DataFrame
        page_size=10,                                              # Show 10 records per page
        style_table={'overflowX': 'auto'},                         # Enable horizontal scrolling
        style_cell={'textAlign': 'left'}                           # Left-align cell text
    ),
    
    html.Br(),
    
    # Visualization charts section
    html.Div([
        dcc.Graph(id='pie-chart'),   # Pie chart for animal type distribution
        dcc.Graph(id='geo-map')      # Geolocation scatter map
    ])
])


In [ ]:
# Cell 4: Dashboard Callbacks
# This callback function is triggered whenever the user changes the
# rescue type dropdown or adjusts the age range slider. It updates
# the data table, pie chart, and geolocation map with filtered data.
#
# Inputs: filter-dropdown value, age-slider value range
# Outputs: data-table records, pie-chart figure, geo-map figure

@app.callback(
    Output('data-table', 'data'),      # Update the data table records
    Output('pie-chart', 'figure'),      # Update the pie chart visualization
    Output('geo-map', 'figure'),        # Update the geolocation map
    Input('filter-dropdown', 'value'),  # Listen for dropdown changes
    Input('age-slider', 'value')        # Listen for slider changes
)
def update_dashboard(selected_filter, age_range):
    """Update all dashboard components based on the selected filter and age range."""
    
    # Look up the pre-filtered DataFrame for the selected rescue type
    filtered_df = filter_map[selected_filter]
    
    # Apply the age range slider as an additional filter on top of the rescue filter
    filtered_df = filtered_df[
        (filtered_df['age_upon_outcome_in_weeks'] >= age_range[0]) &
        (filtered_df['age_upon_outcome_in_weeks'] <= age_range[1])
    ]
    
    # Convert the filtered DataFrame to a list of dictionaries for the data table
    table_data = filtered_df.to_dict('records')
    
    # Create a pie chart showing the distribution of animal types in the filtered data
    pie_fig = px.pie(
        filtered_df,
        names='animal_type',
        title='Animal Type Distribution'
    )
    
    # Create a geolocation scatter map plotting each animal's location
    # Color-coded by animal type with hover data showing breed, age, and outcome
    geo_fig = px.scatter_mapbox(
        filtered_df,
        lat='location_lat',
        lon='location_long',
        color='animal_type',
        hover_name='name',
        hover_data=['breed', 'age_upon_outcome_in_weeks', 'outcome_type'],
        zoom=10,
        height=500
    )
    # Use OpenStreetMap as the map tile layer (free, no API key required)
    geo_fig.update_layout(mapbox_style="open-street-map")
    
    # Return updated data for all three output components
    return table_data, pie_fig, geo_fig


In [ ]:
# Cell 5: Run the Dash Application
# Start the Dash development server on port 8050 with debug mode enabled.
# Debug mode provides auto-reload on code changes and detailed error messages.
if __name__ == "__main__":
    app.run(port=8050, debug=True)  
